### Import libraries

In [1]:
import mediapipe as mp
import cv2
import numpy as np
import os

### Definig Drawing and hand utils

In [2]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

![Image description](../hands.png)


### Basic detection with walk through

In [3]:
cap = cv2.VideoCapture(0)

#
with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.6) as hands: 
    while cap.isOpened():
        ret, frame = cap.read()
        
        # BGR 2 RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Flip on horizontal
        image = cv2.flip(image, 1)
        
        # Set flag
        image.flags.writeable = False
        
        # Detections
        results = hands.process(image)
        
        # Set flag to true
        image.flags.writeable = True
        
        # RGB 2 BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        
        # Rendering results
        if results.multi_hand_landmarks:
            for num, hand in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(image, hand, mp_hands.HAND_CONNECTIONS, 
                                        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                                        mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2, circle_radius=2),
                                         )
            
        
        cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

C:\Users\Hawis\anaconda3\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


### Getting which hand was detected

In [4]:
results.multi_handedness

[classification {
   index: 1
   score: 0.980566561
   label: "Right"
 }]

In [5]:
results.multi_handedness[0].classification[0].label

'Right'

## How to get the index of each part of the hands

#### From the image above you can use the code snippest below to get the various index of corresponding part of the hand 

In [6]:
# Examples below
# Getting the index of the wrist
mp_hands.HandLandmark.WRIST

<HandLandmark.WRIST: 0>

In [7]:
# Getting the index of the middle finger tip
mp_hands.HandLandmark.MIDDLE_FINGER_TIP

<HandLandmark.MIDDLE_FINGER_TIP: 12>

# Getting HandLandmarks

In [8]:
results.multi_hand_landmarks

[landmark {
   x: 0.652756214
   y: 0.663103223
   z: 7.47635113e-007
 }
 landmark {
   x: 0.580905795
   y: 0.644631088
   z: -0.0460661612
 }
 landmark {
   x: 0.513646543
   y: 0.599496841
   z: -0.0755299628
 }
 landmark {
   x: 0.463003427
   y: 0.551291168
   z: -0.100220777
 }
 landmark {
   x: 0.418267965
   y: 0.503996611
   z: -0.124769777
 }
 landmark {
   x: 0.538097
   y: 0.415187299
   z: -0.0560787581
 }
 landmark {
   x: 0.510781467
   y: 0.301490933
   z: -0.0900313333
 }
 landmark {
   x: 0.492651761
   y: 0.231199071
   z: -0.115788259
 }
 landmark {
   x: 0.480844676
   y: 0.169365942
   z: -0.134321168
 }
 landmark {
   x: 0.589049578
   y: 0.387100697
   z: -0.0573635921
 }
 landmark {
   x: 0.580547869
   y: 0.253353357
   z: -0.0855776519
 }
 landmark {
   x: 0.575229824
   y: 0.168100446
   z: -0.108099975
 }
 landmark {
   x: 0.57077086
   y: 0.0971994698
   z: -0.125233337
 }
 landmark {
   x: 0.639658809
   y: 0.389447123
   z: -0.0639923736
 }
 landmark {
 

### How to get coordinates of the parts of the hands

In [9]:
# Getting hand landmarks
results.multi_hand_landmarks[0].landmark[mp_hands.HandLandmark.WRIST]

x: 0.652756214
y: 0.663103223
z: 7.47635113e-007

In [10]:
# Getting hand landmarks
results.multi_hand_landmarks[0].landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP]

x: 0.57077086
y: 0.0971994698
z: -0.125233337

### Simple function to get the coordinates of the parts of the hands specified by user

In [17]:
def get_label(index, hand, results):
    output = None
    for idx, classification in enumerate(results.multi_handedness):
        # if classification.classification[0].index == index:
            
        # Extract Coordinates
        coords = tuple(np.multiply(
            # Change it from wrist if you want the coordinates of a different part of the hand
            np.array((hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].x, hand.landmark[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].y)),
        [640,480]).astype(int))
        
        output = coords

    if output:
        return coords
    else:
        return None

### Testing Functions

In [18]:
cap = cv2.VideoCapture(0)

with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands: 
    while cap.isOpened():
        ret, frame = cap.read()
        
        # BGR 2 RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Flip on horizontal
        image = cv2.flip(image, 1)
        
        # Set flag
        image.flags.writeable = False
        
        # Detections
        results = hands.process(image)
        
        # Set flag to true
        image.flags.writeable = True
        
        # RGB 2 BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Rendering results
        if results.multi_hand_landmarks:
            for num, hand in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(image, hand, mp_hands.HAND_CONNECTIONS, 
                                        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                                        mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2, circle_radius=2),
                                         )
                
                # Render left or right detection
                coords = get_label(num, hand, results)
                if coords:
                    coords = get_label(num, hand, results)
                    cv2.circle(image, coords, 5, (255, 50, 40), 3)
        # Save our image    
        #cv2.imwrite(os.path.join('Output Images', '{}.jpg'.format(uuid.uuid1())), image)
        cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [15]:
results.multi_handedness

[classification {
   index: 1
   score: 0.991788328
   label: "Right"
 }]